# Eksploracyjna Analiza Danych (EDA) - Uzasadnienie Metodologiczne
## Praca Magisterska: Porównanie algorytmów SVD, NCF i LightGCN

Ten notebook realizuje analizę ukierunkowaną na identyfikację wyzwań strukturalnych zbioru MovieLens. Wizualizacje zostały przygotowane w klasycznej, akademickiej kolorystyce (odcienie błękitu i szarości), optymalnej do druku i profesjonalnych publikacji naukowych.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Definicja klasycznej palety akademickiej
ACADEMIC_BLUE = '#1f77b4'  # Głęboki błękit (klasyczny Matplotlib/Tableau)
ACADEMIC_GRAY = '#7f7f7f'  # Neutralna szarość dla tła i osi
LIGHT_BLUE = '#aec7e8'     # Jasny błękit do wypełnień

sns.set_style("ticks") # Klasyczny styl z kreskami na osiach
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'figure.dpi': 100
})

pd.set_option('display.max_columns', None)

# Konfiguracja zbioru danych
DATASET_VERSION = '100k' # '100k' lub '1m'

### 1. Analiza macierzy interakcji i problem rzadkości danych (Sparsity)

In [ ]:
data_dir = '../data'
df_ratings = pd.read_csv(f'{data_dir}/movielens_{DATASET_VERSION}_ratings.csv')

n_users = df_ratings['user_id'].nunique()
n_items = df_ratings['item_id'].nunique()
n_interactions = len(df_ratings)
sparsity = (1 - (n_interactions / (n_users * n_items))) * 100

stats_summary = pd.DataFrame({
    'Parametr': ['Użytkownicy (U)', 'Obiekty (I)', 'Interakcje (E)', 'Rzadkość (S)'],
    'Wartość': [n_users, n_items, n_interactions, f"{sparsity:.2f}%"]
})
display(stats_summary)

### 2. Rozkład Feedbacku i zjawisko "Długiego Ogona" (Long Tail)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 2.1 Popularność filmów
movie_counts = df_ratings.groupby('item_id').size().sort_values(ascending=False).values
ax1.plot(movie_counts, color=ACADEMIC_BLUE, linewidth=2)
ax1.fill_between(range(len(movie_counts)), 0, movie_counts.astype(float), color=ACADEMIC_BLUE, alpha=0.15)
ax1.set_title('Popularność obiektów (Long Tail)')
ax1.set_xlabel('Ranking filmu')
ax1.set_ylabel('Liczba ocen')

# 2.2 Aktywność użytkowników
user_counts = df_ratings.groupby('user_id').size().sort_values(ascending=False).values
ax2.plot(user_counts, color=ACADEMIC_GRAY, linewidth=2)
ax2.fill_between(range(len(user_counts)), 0, user_counts.astype(float), color=ACADEMIC_GRAY, alpha=0.15)
ax2.set_title('Aktywność użytkowników')
ax2.set_xlabel('Ranking użytkownika')
ax2.set_ylabel('Liczba ocen')

plt.tight_layout()
plt.show()

### 3. Identyfikacja problemu "Zimnego Startu" (Cold Start)

In [ ]:
thresholds = [10, 20, 50]
cold_data = []
for t in thresholds:
    u_p = (sum(1 for c in user_counts if c < t) / n_users) * 100
    i_p = (sum(1 for c in movie_counts if c < t) / n_items) * 100
    cold_data.append({'Próg': f'< {t} ocen', 'Użytkownicy (%)': f"{u_p:.1f}%", 'Filmy (%)': f"{i_p:.1f}%"})

display(pd.DataFrame(cold_data))

### 4. Eksploracja metadanych (Side Information dla KGCN)

In [ ]:
df_items = pd.read_csv(f'{data_dir}/movielens_{DATASET_VERSION}_items.csv')

if DATASET_VERSION == '100k':
    genre_cols = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    genre_stats = df_items[genre_cols].sum().sort_values(ascending=False)
else:
    genre_stats = df_items['genres'].str.split('|').explode().value_counts()

plt.figure(figsize=(12, 6))
x_labels = [str(x) for x in genre_stats.index.tolist()]
y_values = genre_stats.values.tolist()

plt.bar(x_labels, y_values, color=ACADEMIC_BLUE, edgecolor=ACADEMIC_GRAY, alpha=0.8)
plt.title('Dystrybucja gatunków (Analiza atrybutów)')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Liczba filmów')
plt.show()

### 5. Poszukiwanie "Szarych Owiec" (Gray Sheep)

In [ ]:
user_variance = df_ratings.groupby('user_id')['rating'].var().fillna(0).sort_values(ascending=False).tolist()

plt.figure(figsize=(10, 5))
plt.hist(user_variance, bins=30, color=LIGHT_BLUE, edgecolor=ACADEMIC_BLUE, alpha=0.8)
plt.axvline(np.mean(user_variance), color='red', linestyle='--', label=f'Średnia: {np.mean(user_variance):.2f}')
plt.title('Rozkład wariancji ocen użytkowników')
plt.xlabel('Wariancja ocen')
plt.ylabel('Liczba użytkowników')
plt.legend()
plt.show()